# Feature Engineering — Cardio AI Platform

This notebook covers:
1. Data cleaning and preprocessing
2. Feature transformations (BMI, pulse pressure, age bins)
3. Feature selection using SelectKBest
4. Saving processed datasets for model training

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded")

## 1. Load Raw Datasets

In [ ]:
heart_df = pd.read_csv("../data/raw/heart_dataset.csv")
cardiac_df = pd.read_csv("../data/raw/cardiac_failure.csv")
framingham_df = pd.read_csv("../data/raw/framingham.csv")

print("Datasets loaded")
print(f"Heart: {heart_df.shape}, Cardiac: {cardiac_df.shape}, Framingham: {framingham_df.shape}")

## 2. Data Cleaning

Handle missing values and drop unnecessary columns.

In [ ]:
# Check missing values across all datasets
for name, df in [("Heart", heart_df), ("Cardiac", cardiac_df), ("Framingham", framingham_df)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"\n{name} — Missing Values:")
        print(missing)
    else:
        print(f"\n{name} — No missing values")

# Drop rows with missing values
heart_clean = heart_df.dropna()
framingham_clean = framingham_df.dropna()

# Drop 'id' column if present in cardiac dataset
cardiac_clean = cardiac_df.copy()
if "id" in cardiac_clean.columns:
    cardiac_clean = cardiac_clean.drop("id", axis=1)
cardiac_clean = cardiac_clean.dropna()

print(f"\nAfter cleaning — Heart: {heart_clean.shape}, Cardiac: {cardiac_clean.shape}, Framingham: {framingham_clean.shape}")

## 3. Feature Engineering — Cardiac Dataset

Create derived features like pulse pressure and BMI.

In [ ]:
# Pulse pressure = systolic - diastolic
cardiac_clean["pulse_pressure"] = cardiac_clean["ap_hi"] - cardiac_clean["ap_lo"]

# BMI from height and weight (height in cm)
cardiac_clean["bmi"] = cardiac_clean["weight"] / ((cardiac_clean["height"] / 100) ** 2)

print("New features added to cardiac dataset:")
print(cardiac_clean[["pulse_pressure", "bmi"]].describe())

## 4. Feature Selection — SelectKBest

In [ ]:
# Feature importance using mutual information for the heart dataset
X_heart = heart_clean.drop("HeartDisease", axis=1)
y_heart = heart_clean["HeartDisease"]

# Only use numeric columns
X_heart_numeric = X_heart.select_dtypes(include=[np.number])

selector = SelectKBest(score_func=mutual_info_classif, k="all")
selector.fit(X_heart_numeric, y_heart)

feature_scores = pd.Series(selector.scores_, index=X_heart_numeric.columns)
feature_scores = feature_scores.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feature_scores.plot(kind="barh", color="#3498db")
plt.title("Feature Importance (Mutual Information) — Heart Dataset")
plt.xlabel("Mutual Information Score")
plt.tight_layout()
plt.show()

## 5. Save Processed Datasets

In [ ]:
import os

os.makedirs("../data/processed", exist_ok=True)

heart_clean.to_csv("../data/processed/heart_clean.csv", index=False)
cardiac_clean.to_csv("../data/processed/cardiac_clean.csv", index=False)
framingham_clean.to_csv("../data/processed/framingham_clean.csv", index=False)

print("All cleaned datasets saved to data/processed/")